# Exercise 2.1 (optional) - Takahashi adder

In-place ripple-carry adder from Takahashi, Tani, Kunihiro (https://arxiv.org/abs/0910.2530), without carry-in:

$$|a\rangle|b\rangle|z\rangle \rightarrow |a\rangle|a+b \bmod 2^n\rangle|z \oplus s_n\rangle$$

The sum overwrites `b`, the carry-out goes into the single ancilla `z`. For $n = 2$ only $2n + 1 = 5$ qubits are needed.

Steps (qubit 0 = LSB):
1. CNOT(a_i -> b_i) for i = 1..n-1
2. CNOT(a_{n-1} -> z), then CNOT(a_i -> a_{i+1}) for i = n-2..1
3. Toffoli(b_i, a_i -> a_{i+1}) for i = 0..n-2
4. Toffoli(b_{n-1}, a_{n-1} -> z)
5. for i = n-1..1: CNOT(a_i -> b_i), Toffoli(b_{i-1}, a_{i-1} -> a_i)
6. CNOT(a_i -> a_{i+1}) for i = 1..n-2
7. CNOT(a_i -> b_i) for i = 0..n-1

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
import matplotlib.pyplot as plt


def takahashi_adder(n):
    a = QuantumRegister(n, "a")
    b = QuantumRegister(n, "b")
    z = QuantumRegister(1, "z")
    qc = QuantumCircuit(a, b, z, name="ADD")

    # Step 1
    for i in range(1, n):
        qc.cx(a[i], b[i])

    # Step 2
    qc.cx(a[n - 1], z[0])
    for i in range(n - 2, 0, -1):
        qc.cx(a[i], a[i + 1])

    # Step 3
    for i in range(n - 1):
        qc.ccx(b[i], a[i], a[i + 1])

    # Step 4
    qc.ccx(b[n - 1], a[n - 1], z[0])

    # Step 5
    for i in range(n - 1, 0, -1):
        qc.cx(a[i], b[i])
        qc.ccx(b[i - 1], a[i - 1], a[i])

    # Step 6
    for i in range(1, n - 1):
        qc.cx(a[i], a[i + 1])

    # Step 7
    for i in range(n):
        qc.cx(a[i], b[i])

    return qc


n = 2
adder = takahashi_adder(n)

fig = adder.draw(output="mpl")
display(fig)
plt.close(fig)

## Verification of all input combinations

In [ ]:
from qiskit.quantum_info import Statevector


def init_inputs(qc, a_val, b_val, n):
    for i in range(n):
        if (a_val >> i) & 1:
            qc.x(i)
        if (b_val >> i) & 1:
            qc.x(n + i)


all_ok = True
for a_val in range(2**n):
    for b_val in range(2**n):
        qc = QuantumCircuit(2 * n + 1)
        init_inputs(qc, a_val, b_val, n)
        qc.compose(adder, inplace=True)

        # The output is a basis state: read it from the most probable outcome
        # (bitstring is little-endian: z b_{n-1}..b_0 a_{n-1}..a_0)
        out = Statevector(qc).probabilities_dict()
        bits = max(out, key=out.get)
        a_out = int(bits[-n:], 2)
        s_out = int(bits[: n + 1], 2)

        ok = a_out == a_val and s_out == a_val + b_val
        all_ok &= ok
        print(f"a={a_val:0{n}b} b={b_val:0{n}b} -> sum={s_out:0{n + 1}b} ({s_out})  a={a_out:0{n}b}  {'OK' if ok else 'ERROR'}")

print("\nAll combinations correct:", all_ok)

## Submission inputs: a = 01, b = 11

In [ ]:
from qiskit.providers.basic_provider import BasicProvider

a_in = 0b01
b_in = 0b11

a = QuantumRegister(n, "a")
b = QuantumRegister(n, "b")
z = QuantumRegister(1, "z")
s = ClassicalRegister(n + 1, "s")
qc = QuantumCircuit(a, b, z, s)

init_inputs(qc, a_in, b_in, n)
qc.barrier()
qc.compose(adder, inplace=True)
qc.barrier()

# Sum = z b_{n-1} .. b_0
qc.measure(list(b) + [z[0]], s)

fig = qc.draw(output="mpl", plot_barriers=False)
display(fig)
plt.close(fig)

In [ ]:
backend = BasicProvider().get_backend("basic_simulator")
result = backend.run(qc).result()

counts = result.get_counts()
print(counts)
print(f"{a_in} + {b_in} = {int(max(counts, key=counts.get), 2)}")